# FLOWTRUST-AFR — Training 01 / T01-A0 Scene Gate
**Date:** 12 August 2026

Goal: make the visual pipeline reject ordinary rooms, people, walls and other non-conveyor scenes **before** any AFR analysis.

This bootstrap GPU run uses:
- CoalAD conveyor-belt coal scenes as positive examples;
- COCO 2017 validation images as broad non-conveyor hard negatives;
- a 304M-parameter DINOv2-L backbone immediately, or the 300M DINOv3 ViT-L backbone when its model access is available.

**Acceptance gate:** false conveyor acceptance rate <= 2% on a group-disjoint closed test.

This is Training 01, not the final vision model. Industrial non-conveyor hard negatives and the iron-ore/CDW conveyor datasets are added in the next expansion run.


In [ ]:
import os, sys, subprocess, torch, platform
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU: Runtime > Change runtime type > Hardware accelerator > GPU")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
!rm -rf /content/SIC2026
!git clone --depth 1 --branch v2-model-rebuild https://github.com/diawbirane10-lgtm/SIC2026.git /content/SIC2026
%cd /content/SIC2026
!pip -q install "transformers>=4.56" "scikit-learn>=1.6" "pandas>=2.2" "joblib>=1.4" "Pillow>=11" "wget>=3.2"


In [ ]:
from pathlib import Path
import os, tarfile, urllib.request, zipfile

ROOT = Path("/content/flowtrust_data")
ROOT.mkdir(parents=True, exist_ok=True)

coal_urls = {
    "train": "https://www.modelscope.cn/datasets/lyfjwp/CoalAD/resolve/master/coal_ad_train.tar",
    "test": "https://www.modelscope.cn/datasets/lyfjwp/CoalAD/resolve/master/coal_ad_test.tar",
}
for split, url in coal_urls.items():
    tar_path = ROOT / f"coal_ad_{split}.tar"
    if not (ROOT / "coal_ad" / split).exists():
        print("Downloading", url)
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path) as tf:
            tf.extractall(ROOT)
        tar_path.unlink(missing_ok=True)

print("CoalAD ready:", ROOT / "coal_ad")


In [ ]:
# Broad hard negatives: COCO 2017 validation images.
# We do NOT redistribute these images or include them in GitHub artifacts.
import urllib.request, zipfile
coco_zip = ROOT / "val2017.zip"
coco_dir = ROOT / "coco"
coco_dir.mkdir(exist_ok=True)
if not (coco_dir / "val2017").exists():
    print("Downloading COCO val2017 (~780 MB)...")
    urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", coco_zip)
    with zipfile.ZipFile(coco_zip) as zf:
        zf.extractall(coco_dir)
    coco_zip.unlink(missing_ok=True)
print("COCO ready:", coco_dir / "val2017")


In [ ]:
!python v2/training/build_t01a_bootstrap_manifest.py \
  --coalad /content/flowtrust_data/coal_ad \
  --negative-root /content/flowtrust_data/coco/val2017 \
  --out /content/SIC2026/data/manifests/t01a_bootstrap.csv \
  --max-positive 3500 \
  --max-negative 3500

import pandas as pd
m = pd.read_csv("/content/SIC2026/data/manifests/t01a_bootstrap.csv")
display(m.groupby(["source","label"]).size().rename("images").to_frame())
print("Total:", len(m), "Groups:", m.group.nunique())


In [ ]:
# Immediate public 304M fallback. Change to DINOv3 when you have accepted its model access.
MODEL_ID = "facebook/dinov2-large"

# If a Hugging Face token with accepted DINOv3 access is present, prefer DINOv3 ViT-L automatically.
import os
if os.environ.get("HF_TOKEN"):
    MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"

print("Training backbone:", MODEL_ID)


In [ ]:
!rm -rf /content/SIC2026/artifacts/t01/vision_scene_gate
!python v2/training/t01_scene_gate.py \
  --manifest /content/SIC2026/data/manifests/t01a_bootstrap.csv \
  --outdir /content/SIC2026/artifacts/t01/vision_scene_gate \
  --model "$MODEL_ID" \
  --batch-size 8 \
  --seed 2026


In [ ]:
import json, pandas as pd
metrics_path = "/content/SIC2026/artifacts/t01/vision_scene_gate/metrics.json"
metrics = json.load(open(metrics_path))
print(json.dumps({
    "model": metrics["model"],
    "parameters": metrics["backbone_parameters"],
    "macro_f1": metrics["macro_f1"],
    "ece": metrics["ece"],
    "false_conveyor_accept_rate": metrics["false_conveyor_accept_rate"],
    "gate_passed": metrics["gate_passed"],
}, indent=2))
pred = pd.read_csv("/content/SIC2026/artifacts/t01/vision_scene_gate/closed_test_predictions.csv")
display(pred[pred.label != pred.prediction].head(30))


## Tablet sanity test
Optional but strongly recommended: upload 3–10 photos from the Xiaomi Pad 6, including:
1. the room/wall case that previously produced the absurd “86/100” result;
2. a person/object/non-industrial scene;
3. a conveyor image displayed on another screen.

These images are evaluated locally in this Colab session.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
# Evaluate uploaded images with the trained scene gate.
from pathlib import Path
import joblib, torch, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

bundle = joblib.load("/content/SIC2026/artifacts/t01/vision_scene_gate/scene_gate_head.joblib")
model_id = bundle["model_id"]
processor = AutoImageProcessor.from_pretrained(model_id)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
backbone = AutoModel.from_pretrained(model_id, torch_dtype=dtype).cuda().eval()

rows = []
with torch.inference_mode():
    for name in uploaded:
        img = Image.open(name).convert("RGB")
        inp = {k:v.cuda() for k,v in processor(images=[img], return_tensors="pt").items()}
        out = backbone(**inp)
        emb = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state.mean(1)
        proba = bundle["classifier"].predict_proba(emb.float().cpu().numpy())[0]
        idx = int(proba.argmax())
        label = bundle["label_encoder"].inverse_transform([idx])[0]
        rows.append((name, label, float(proba[idx])))
        
import pandas as pd
display(pd.DataFrame(rows, columns=["image","prediction","confidence"]))


In [ ]:
# Package only metrics/model head/predictions — never the downloaded datasets.
import shutil, os
archive = shutil.make_archive(
    "/content/FLOWTRUST_T01A_20260812",
    "zip",
    "/content/SIC2026/artifacts/t01/vision_scene_gate"
)
print("Artifact:", archive)
from google.colab import files
files.download(archive)
